# A1 Plate Detection — Training Notebook (YOLOv8 / YOLO26)

**UIT Graduation Thesis 2026** — Smart Parking Management System using Computer Vision and Edge AI

Trains 2 model families (`yolov8n`, `yolo26n`) across 3 seeds on the A1 dataset (2-class: `bien_1hang`, `bien_2hang`).  
All logic lives in the `plate_detect` package; this notebook only orchestrates.

**How to run:** Select a Colab GPU runtime → Run All. Resumable via Drive checkpoints (each seed writes to Drive; skip already-completed seeds by re-running from cell 9).

In [ ]:
# ── Config — edit here, then Run All ────────────────────────────────────────
DRIVE_ROOT = "/content/drive/MyDrive/UIT_2026"
REPO_URL   = "https://github.com/UIT-DoAnCuoiKi/UIT2026-DoAnCuoiKi.git"
MODELS     = ["yolov8n", "yolo26n"]
SEEDS      = [0, 1, 2]

In [ ]:
# ── 1. Pinned install ─────────────────────────────────────────────────────
!pip install -q ultralytics==8.4.37
import ultralytics
ultralytics.checks()

In [ ]:
# ── 2. GPU check ──────────────────────────────────────────────────────────
import torch
assert torch.cuda.is_available(), "Select a Colab GPU runtime (not TPU/CPU)."
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# ── 3. Clone repo + install plate_detect package ─────────────────────────
import os
if not os.path.exists("/content/repo"):
    !git clone -q $REPO_URL /content/repo
!pip install -q -e /content/repo/src/ml/plate_detect

In [ ]:
# ── 4. Mount Drive (checkpoints + weights land here) ─────────────────────
from google.colab import drive
drive.mount("/content/drive")
import os
os.makedirs(f"{DRIVE_ROOT}/runs", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/weights", exist_ok=True)
print("Drive mounted. Runs →", f"{DRIVE_ROOT}/runs")

In [ ]:
!pip install -q kaggle
# ── 5. Pull A1 dataset via Kaggle API ────────────────────────────────────
# Requires ~/.kaggle/kaggle.json — upload it via the Colab sidebar first, e.g.:
#   from google.colab import files; files.upload()  # then move to ~/.kaggle/
import os
os.makedirs("/root/.kaggle", exist_ok=True)
# Uncomment and run once if kaggle.json was just uploaded:
# import shutil; shutil.copy("/content/kaggle.json", "/root/.kaggle/kaggle.json")
# !chmod 600 /root/.kaggle/kaggle.json

!kaggle datasets download -d duydieunguyen/licenseplates \
    -p /content/data/raw/kaggle_vn_plate_segment --unzip
print("Dataset ready at /content/data/raw/kaggle_vn_plate_segment")

In [ ]:
# ── 6. Build Config + prepare dataset ────────────────────────────────────
from plate_detect.config import Config
from plate_detect.data.prepare import prepare

cfg = Config(
    raw_dir="/content/data/raw/kaggle_vn_plate_segment",
    processed_dir="/content/data/processed/a1_det",
    dataset_yaml="/content/repo/src/ml/plate_detect/configs/a1_det.yaml",
    split_dir="/content/repo/src/ml/plate_detect/configs/split",
)
result = prepare(cfg)
print("prepare →", result)

In [ ]:
from plate_detect.train.trainer import run_train
from plate_detect.eval.evaluate import aggregate_seeds, append_experiment
from ultralytics import YOLO
import os, shutil

FIG_DIR = "/content/repo/docs/report/figures"
os.makedirs(FIG_DIR, exist_ok=True)

runs = {}
best_runs = {}   # model_key -> (best_seed_index, best_run_dir)
for mk in MODELS:
    seed_metrics, seed_dirs = [], []
    for s in SEEDS:
        rd = run_train(mk, cfg, cfg.dataset_yaml, seed=s, project=f"{DRIVE_ROOT}/runs")
        seed_dirs.append(rd)
        m = YOLO(f"{rd}/weights/best.pt").val(data=cfg.dataset_yaml, split="test")
        for fig in ("confusion_matrix.png", "results.png", "PR_curve.png"):
            src = os.path.join(rd, fig)
            if os.path.exists(src):
                shutil.copy(src, os.path.join(FIG_DIR, f"{mk}_s{s}_{fig}"))
        seed_metrics.append({"map50": m.box.map50, "map5095": m.box.map,
                             "precision": m.box.mp, "recall": m.box.mr})
    runs[mk] = aggregate_seeds(seed_metrics)
    best_seed = max(range(len(SEEDS)), key=lambda i: seed_metrics[i]["map50"])
    best_runs[mk] = (best_seed, seed_dirs[best_seed])
    append_experiment("/content/repo/src/ml/experiments.csv", mk, "A1",
                      f"imgsz={cfg.imgsz};epochs={cfg.epochs};seeds={SEEDS}",
                      seed_metrics[best_seed], f"weights/{mk}_a1_s{best_seed}.pt")
print(runs)

In [ ]:
from plate_detect.export.to_onnx import export
import os, shutil

WEIGHTS_DIR = "/content/repo/src/ml/plate_detect/weights"
os.makedirs(WEIGHTS_DIR, exist_ok=True)
for mk, (best_seed, rd) in best_runs.items():
    best_pt = f"{rd}/weights/best.pt"
    dst_pt = os.path.join(WEIGHTS_DIR, f"{mk}_a1_s{best_seed}.pt")
    shutil.copy(best_pt, dst_pt)
    onnx_path = export(best_pt, os.path.join(WEIGHTS_DIR, f"{mk}_a1.onnx"), imgsz=cfg.imgsz)
    print("exported", dst_pt, "->", onnx_path)

## Next Steps — copy weights and commit (from your local machine)

After the training run finishes:

1. **Download weights from Drive** (`$DRIVE_ROOT/weights/`):
   - `yolov8n_a1_best.pt` + `yolov8n_a1_best.onnx`
   - `yolo26n_a1_best.pt` + `yolo26n_a1_best.onnx`

2. **Place them in `src/ml/plate_detect/weights/`** (tracked by git-lfs — see `.gitattributes`).

3. **Commit from your local machine:**
   ```bash
   git add src/ml/plate_detect/weights/
   git add src/ml/experiments.csv
   git add docs/report/figures/
   git commit -m "feat(plate_detect): add trained A1 weights and eval figures"
   git push
   ```

**Success gate:** mAP@0.5 >= 0.90 on the test split for both model families.  
If below threshold: increase `epochs` in `Config`, add more augmentation, or collect more data.